In [1]:
import pandas as pd


In [2]:
df = pd.read_csv('deep_learning_dataset.csv')

In [3]:
df


,age,gender,bmi,blood_pressure,cholesterol,glucose,smoking,alcohol,exercise_hours,sleep_hours,...,family_history,heart_rate,diet_score,num_medications,previous_stroke,income_level,education_years,water_intake,fruit_veg_score,heart_risk_score
0,56,0,18.8,87,279,252,0,0,8.1,5.0,...,1,98,9,6,0,2,14,3.6,5,31.94
1,69,1,23.0,126,370,203,0,0,6.1,4.4,...,0,72,2,6,0,1,12,3.9,4,49.75
2,46,1,40.1,130,189,152,1,0,8.8,4.7,...,1,73,7,4,0,3,9,3.6,8,30.23
3,32,1,24.8,73,187,291,0,1,0.6,4.6,...,1,113,1,0,0,3,9,3.2,3,56.46
4,60,0,29.6,138,182,197,1,0,9.1,7.9,...,0,103,8,6,0,3,9,2.2,3,29.26
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,55,0,43.0,173,216,268,0,1,0.2,6.6,...,0,74,4,1,0,1,18,3.6,9,79.40
9996,51,1,24.3,168,229,188,0,1,2.8,9.8,...,0,77,8,5,0,3,8,0.5,7,39.24
9997,57,1,25.8,128,219,190,1,0,1.4,8.6,...,1,64,1,1,0,1,11,1.4,9,60.51
9998,64,0,29.2,115,312,171,0,0,6.2,7.2,...,1,116,2,7,0,3,21,3.9,6,47.42


In [4]:
from sklearn.model_selection import train_test_split

In [5]:
x = df.drop(columns = 'smoking')
y = df['smoking']


In [6]:
x_train , x_test , y_train , y_test = train_test_split(x,y,test_size = 0.2 , random_state = 42)

In [62]:
import tensorflow
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense , Dropout, BatchNormalization
from keras.callbacks import EarlyStopping

In [10]:
import kerastuner as kt


/tmp/ipykernel_2958/1337601089.py:1: DeprecationWarning: `import kerastuner` is deprecated, please use `import keras_tuner`.
  import kerastuner as kt


hidden layers
neurons
activation function
dropout
optimizer



In [12]:
#hidden layers
#neurons
#activation function
#dropout
#optimizer

In [82]:
from keras.src.models import model
def build_model(hp):

  counter = 0
  model = Sequential()
  for i in range(hp.Int('hidden_layers', min_value= 1 , max_value = 10)):

          if counter ==0:

              model.add(Dense(units = hp.Int('unit' + str(i) , min_value = 8 , max_value = 128),
                              activation = hp.Choice('activation' + str(i), values = ['relu', 'tanh', 'sigmoid']),
                              input_dim = 20
                              ))

              model.add(Dropout(hp.Choice('dropout'+str(i),values = [0.1,0.2,0.3,0.4,0.5])))
              counter +=1

          else:
              model.add(Dense(units = hp.Int('unit' + str(i), min_value = 8 , max_value = 128),
                               activation = hp.Choice('activation'+str(i), values = ['relu', 'tanh','sigmoid'])))

              model.add(Dropout(hp.Choice('dropout'+str(i), values = [0.1,0.2,0.3,0.4,0.5])))

  model.add(Dense(1, activation = 'sigmoid'))

  model.compile(optimizer = hp.Choice('optimizer', values = ['adam','rmsprop','sgd']), loss = 'binary_crossentropy', metrics = ['accuracy'])
  return model



In [83]:
tuners = kt.RandomSearch(build_model, objective = 'val_accuracy', max_trials = 10)

Reloading Tuner from ./untitled_project/tuner0.json


In [86]:
tuners.search(x_train, y_train, epochs = 10, validation_data = (x_test, y_test))

In [88]:
tuners.get_best_hyperparameters()[0].values

{'hidden_layers': 8,
 'unit0': 91,
 'activation0': 'relu',
 'dropout0': 0.8,
 'optimizer': 'sgd',
 'activation1': 'leaky_relu',
 'unit1': 30,
 'dropout1': 0.3,
 'unit2': 101,
 'activation2': 'relu',
 'dropout2': 0.3,
 'unit3': 35,
 'activation3': 'tanh',
 'dropout3': 0.1,
 'unit4': 34,
 'activation4': 'relu',
 'dropout4': 0.4,
 'unit5': 85,
 'activation5': 'tanh',
 'dropout5': 0.9,
 'unit6': 60,
 'activation6': 'tanh',
 'dropout6': 0.3,
 'unit7': 125,
 'activation7': 'relu',
 'dropout7': 0.8,
 'unit8': 84,
 'activation8': 'relu',
 'dropout8': 0.3,
 'unit9': 57,
 'activation9': 'tanh',
 'dropout9': 0.6}

In [89]:
callback = EarlyStopping(monitor = 'val_loss', patience = 10)

In [90]:
model = tuners.get_best_models(num_models = 1)[0]

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [92]:
model.fit(x_train, y_train , epochs = 100 , validation_data = (x_test, y_test),initial_epoch=5, callbacks = callback, batch_size=100)

Epoch 6/100
80/80 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.6019 - loss: 0.6971 - val_accuracy: 0.6640 - val_loss: 0.6600
Epoch 7/100
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6077 - loss: 0.6874 - val_accuracy: 0.6640 - val_loss: 0.6583
Epoch 8/100
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6223 - loss: 0.6763 - val_accuracy: 0.6640 - val_loss: 0.6599
Epoch 9/100
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6324 - loss: 0.6719 - val_accuracy: 0.6640 - val_loss: 0.6596
Epoch 10/100
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6342 - loss: 0.6637 - val_accuracy: 0.6640 - val_loss: 0.6589
Epoch 11/100
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6425 - loss: 0.6614 - val_accuracy: 0.6640 - val_loss: 0.6586
Epoch 12/100
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6474 - loss: 0.6591 - val_accuracy: 0.6640 - val_loss: 0.6567
Epoch 13/100
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6488 - loss: 0.6541 - val_accuracy: 0.6640